In [ ]:
from google.colab import drive
import os
import zipfile

# 1. Mount the Drive
drive.mount('/content/drive')

# 2. Define our paths
archive_path = "/content/drive/MyDrive/VictorianGPT_Master_Archive.zip"
unzipped_destination = "/content/drive/MyDrive/VictorianGPT"

# 3. Unpack the new 40-book vault (if it exists)
if os.path.exists(archive_path):
    print("Found the 40-book master archive! Unpacking to Drive (this may take a moment)...")
    with zipfile.ZipFile(archive_path, 'r') as zip_ref:
        zip_ref.extractall("/content/drive/MyDrive/")
    print("Archive unpacked! The old 4-book folder has been successfully overwritten.")
else:
    print("No zip archive found. Assuming the folder was transferred via copytree.")

# Now verify what is actually in the folder Notebook 2 is about to read!
cleaned_dir = f"{unzipped_destination}/cleaned"
if os.path.exists(cleaned_dir):
    book_count = len([f for f in os.listdir(cleaned_dir) if f.endswith('.txt')])
    print(f"\nSUCCESS: Notebook 2 sees {book_count} books ready for training!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Found the 40-book master archive! Unpacking to Drive (this may take a moment)...
Archive unpacked! The old 4-book folder has been successfully overwritten.

SUCCESS: Notebook 2 sees 40 books ready for training!


In [ ]:
import shutil
import zipfile
import os
from google.colab import drive

# Connect to Drive
drive.mount('/content/drive')

old_folder = "/content/drive/MyDrive/VictorianGPT"
archive_path = "/content/drive/MyDrive/VictorianGPT_Master_Archive.zip"

# 1. Nuke the old 4-book folder
if os.path.exists(old_folder):
    print("Trashing the old 4-book folder...")
    shutil.rmtree(old_folder)
    print("Old folder deleted.")

# 2. Extract the new 40-book archive directly into the expected path
if os.path.exists(archive_path):
    print("Unpacking the new 40-book zip file...")
    os.makedirs(old_folder, exist_ok=True)

    with zipfile.ZipFile(archive_path, 'r') as zip_ref:
        # We extract the contents directly into the new VictorianGPT folder
        zip_ref.extractall(old_folder)

    print("\nExtraction complete! Your 40-book corpus is now correctly placed.")

    # Let's verify it worked by checking the 'cleaned' folder count
    cleaned_dir = f"{old_folder}/cleaned"
    if os.path.exists(cleaned_dir):
        book_count = len([f for f in os.listdir(cleaned_dir) if f.endswith('.txt')])
        print(f"SUCCESS: Notebook 2 now sees {book_count} books ready for training!")
else:
    print(f"Error: Could not find {archive_path}. Ensure Notebook 1 finished saving the zip to your Drive.")

Mounted at /content/drive
Trashing the old 4-book folder...
Old folder deleted.
Unpacking the new 40-book zip file...

Extraction complete! Your 40-book corpus is now correctly placed.
SUCCESS: Notebook 2 now sees 40 books ready for training!


In [ ]:
from google.colab import drive
import os

print("Requesting access to the Google Drive vault...")
drive.mount('/content/drive')

PROJECT_PATH = "/content/drive/MyDrive/VictorianGPT"
clean_folder = f"{PROJECT_PATH}/cleaned"
dialogue_folder = f"{PROJECT_PATH}/dialogues"

os.makedirs(dialogue_folder, exist_ok=True)
print(f"Directories verified. Base project path: {PROJECT_PATH}")

Requesting access to the Google Drive vault...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Directories verified. Base project path: /content/drive/MyDrive/VictorianGPT


In [ ]:
import re
import json
import os

print("Commencing extraction from authentic manuscripts...")

authentic_pairs = []
total_quotes = 0

for file in os.listdir(clean_folder):
    if file.endswith(".txt"):
        with open(os.path.join(clean_folder, file), 'r', encoding="utf8") as f:
            text = f.read()

        # Extract from this specific book
        raw_dialogues = re.findall(r'["“](.*?)["”]', text, re.DOTALL)

        book_dialogues = []
        for d in raw_dialogues:
            clean_quote = d.replace('\n', ' ').strip()
            clean_quote = re.sub(r'\s+', ' ', clean_quote)
            if len(clean_quote) > 5:
                book_dialogues.append(clean_quote)

        total_quotes += len(book_dialogues)

        # Map consecutive quotes as conversation pairs
        for i in range(len(book_dialogues)-1):
            input_text = book_dialogues[i]
            output_text = book_dialogues[i+1]

            # Enforce minimum length to prevent junk data
            if len(input_text) > 15 and len(output_text) > 15:
                authentic_pairs.append({
                    "input": input_text,
                    "output": output_text,
                    "source": "authentic_novel"
                })

print(f"Total individual quotes extracted: {total_quotes}")
print(f"Total authentic conversational pairs created: {len(authentic_pairs)}")

Commencing extraction from authentic manuscripts...
Total individual quotes extracted: 64757
Total authentic conversational pairs created: 45881


In [ ]:
print("Preparing synthetic emotional states...")

emotions = ["sad", "happy", "lonely", "anxious", "stressed", "worried", "confused", "excited", "angry"]
situations = ["after exams", "after an interview", "after bad news", "while studying", "after losing something", "during college", "after an argument", "after a long day"]
actions = ["I feel", "I became", "I am", "I suddenly feel", "I think I am", "I am feeling"]

all_inputs = []
for a in actions:
    for e in emotions:
        for s in situations:
            all_inputs.append(f"{a} {e} {s}")

print(f"Generated {len(all_inputs)} modern conversational prompts for Qwen.")

Preparing synthetic emotional states...
Generated 432 modern conversational prompts for Qwen.


In [ ]:
# Install dependencies if not present
!pip install -q transformers accelerate sentencepiece datasets
from transformers import pipeline
from tqdm import tqdm

print("Loading Qwen 3B model into GPU...")
generator = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-3B-Instruct",
    device_map="auto",
    batch_size=8 # Crucial for processing hundreds of prompts swiftly
)
print("Model loaded.")

# Build the exact prompts
prompts = []
for text in all_inputs:
    prompt = f"""You are converting modern English into natural Victorian English.
Rules:
- Speak like an educated person from late 19th-century England
- Avoid Shakespeare completely
- Never use: thou,thee,thy,dost,hast,methinks,yea
- Keep replies concise (1–3 sentences)
- Do not invent fictional situations
- Do not pretend to physically accompany the user
- Never explain your writing choices
- Never say: "feel free to modify"
- Reply naturally as a chatbot

Modern sentence:
{text}

Victorian response:"""
    prompts.append(prompt)

synthetic_pairs = []

print("Generating synthetic Victorian responses...")
# Process in batches using the pipeline
outputs = generator(
    prompts,
    max_new_tokens=50,
    temperature=0.7,
    do_sample=True,
    top_p=0.9
)

# Parse the outputs safely
for original_text, out in zip(all_inputs, outputs):
    raw_response = out[0]["generated_text"]

    # Safely split based on the prompt's trigger word
    if "Victorian response:" in raw_response:
        clean_response = raw_response.split("Victorian response:")[-1].strip()
    else:
        clean_response = raw_response.strip()

    synthetic_pairs.append({
        "input": original_text,
        "output": clean_response,
        "source": "synthetic_qwen"
    })

print(f"Successfully generated {len(synthetic_pairs)} synthetic pairs.")

Loading Qwen 3B model into GPU...


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'top_p', 'do_sample', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Model loaded.
Generating synthetic Victorian responses...


[transformers] Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=50) and `max_length

Successfully generated 432 synthetic pairs.


In [ ]:
import json

# Combine both datasets
master_dataset = authentic_pairs + synthetic_pairs

print(f"Authentic Pairs: {len(authentic_pairs)}")
print(f"Synthetic Pairs: {len(synthetic_pairs)}")
print(f"Total Master Dataset Size: {len(master_dataset)}")

# Save to the final JSON file
save_path = f"{dialogue_folder}/victorian_dataset_master.json"

with open(save_path, "w", encoding="utf-8") as f:
    json.dump(master_dataset, f, indent=4)

print(f"\nSuccess! The master production dataset has been safely archived at:\n{save_path}")

Authentic Pairs: 45881
Synthetic Pairs: 432
Total Master Dataset Size: 46313

Success! The master production dataset has been safely archived at:
/content/drive/MyDrive/VictorianGPT/dialogues/victorian_dataset_master.json
